[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_06_deep_learning/ch32_attention_transformers_modern_ai/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 32 Walkthrough: Attention, Transformers, Fine-Tuning & Modern Foundation Models

**Book Reference:** Chapter 32 (Part VI Capstone), Sections 32.1 – 32.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every component—Scaled Dot-Product Attention, Sinusoidal Positional Encoding, Multi-Head Self-Attention, Transformer Encoder Block, Clinical Triage Note Classifier, and LoRA (`Delta W = B @ A`)—is implemented 100% natively in PyTorch so all cells run offline on CPU in seconds with zero external downloads.

In [ ]:
# @title Step 0: Click Play (▶) to Load PyTorch, NumPy & Scikit-Learn
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import math
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import sharukh_tutor as tutor

torch.manual_seed(42)
torch.set_num_threads(1)
np.random.seed(42)
print("PyTorch:", torch.__version__, "| Device: cpu (100% offline)")


## Section 32.3.1 — Why $\sqrt{d_k}$ Prevents Softmax Gradient Saturation

In [ ]:
torch.manual_seed(42)
d_k = 64
q_demo = torch.randn(1, 6, d_k)
k_demo = torch.randn(1, 6, d_k)

raw_scores = q_demo @ k_demo.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(d_k)

raw_attn = torch.softmax(raw_scores, dim=-1)
scaled_attn = torch.softmax(scaled_scores, dim=-1)

print(f"Unscaled score std: {raw_scores.std().item():.3f} | "
      f"max weight: {raw_attn.max().item():.4f}")
print(f"Scaled   score std: {scaled_scores.std().item():.3f} | "
      f"max weight: {scaled_attn.max().item():.4f}")


## Section 32.3.2 — Scaled Dot-Product Attention & Causal Masking

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention (mask: 1=allow, 0=block)."""
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V, attn

torch.manual_seed(7)
Q_ex = torch.randn(1, 4, 8)
K_ex = torch.randn(1, 4, 8)
V_ex = torch.randn(1, 4, 8)
causal_mask = torch.tril(torch.ones(1, 4, 4))
ctx, causal_attn = scaled_dot_product_attention(
    Q_ex, K_ex, V_ex, mask=causal_mask
)
upper_max = torch.triu(causal_attn[0], diagonal=1).max().item()
row_err = (causal_attn.sum(dim=-1) - 1.0).abs().max().item()
print("Context shape:", tuple(ctx.shape),
      "| Attn shape:", tuple(causal_attn.shape))
print("Causal attention matrix (rounded):")
print(np.round(causal_attn[0].detach().numpy(), 3))
print(f"Upper-triangle max: {upper_max:.6f} | "
      f"Row-sum err: {row_err:.6f}")


## Section 32.3.3 — Sinusoidal Positional Encoding (Vaswani et al., 2017)

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Return (max_len, d_model) float32 sinusoidal position table."""
    pe = torch.zeros(max_len, d_model, dtype=torch.float32)
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term[: d_model // 2])
    return pe

pe = sinusoidal_positional_encoding(max_len=12, d_model=16)
cos = nn.CosineSimilarity(dim=0)
print("PE shape:", tuple(pe.shape), "| dtype:", pe.dtype)
print("pos=0 first 6 dims:", np.round(pe[0, :6].numpy(), 3))
print(f"cos(pos0, pos1): {cos(pe[0], pe[1]).item():.4f} | "
      f"cos(pos0, pos10): {cos(pe[0], pe[10]).item():.4f}")


## Section 32.3.4 — Multi-Head Self-Attention & Transformer Encoder Block

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention with W_q, W_k, W_v, W_o projections."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(
                f"d_model ({d_model}) must be divisible by "
                f"num_heads ({num_heads})"
            )
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        H, dk = self.num_heads, self.d_k
        Q = self.W_q(x).view(B, T, H, dk).transpose(1, 2)
        K = self.W_k(x).view(B, T, H, dk).transpose(1, 2)
        V = self.W_v(x).view(B, T, H, dk).transpose(1, 2)
        ctx, attn = scaled_dot_product_attention(Q, K, V, mask=mask)
        merged = ctx.transpose(1, 2).contiguous().view(
            B, T, self.d_model
        )
        return self.W_o(merged), attn


class TransformerEncoderBlock(nn.Module):
    """Pre/Post-LN Transformer Encoder: MHA + Residual + GELU FFN."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, mask=None):
        attn_out, weights = self.attn(x, mask=mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x, weights

torch.manual_seed(42)
block = TransformerEncoderBlock(d_model=16, num_heads=4, d_ff=32)
block.eval()
x_in = torch.randn(2, 6, 16)
pad_m = torch.tensor(
    [[1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 0]], dtype=torch.float32
).view(2, 1, 1, 6)
h_out, w_out = block(x_in, mask=pad_m)
print("Block output:", tuple(h_out.shape),
      "| Attention:", tuple(w_out.shape))
print("Max attention on <PAD> tokens:",
      round(float(w_out[0, :, :4, 4:].max().item()), 6))


## Section 32.3.5 — Clinical Triage Note Classifier & Attention Interpretability

In [ ]:
class ClinicalVocabTokenizer:
    """Deterministic whitespace clinical tokenizer with PAD & UNK."""

    PAD_ID = 0
    UNK_ID = 1

    def __init__(self):
        self.token_to_id = {"<PAD>": 0, "<UNK>": 1}
        self.id_to_token = {0: "<PAD>", 1: "<UNK>"}

    def fit(self, texts):
        words = sorted(
            {w for t in texts for w in t.lower().split()}
        )
        self.token_to_id = {"<PAD>": 0, "<UNK>": 1}
        for idx, word in enumerate(words, start=2):
            self.token_to_id[word] = idx
        self.id_to_token = {
            v: k for k, v in self.token_to_id.items()
        }
        return self

    def encode(self, text, max_len):
        ids = [
            self.token_to_id.get(w, self.UNK_ID)
            for w in text.lower().split()
        ][:max_len]
        return ids + [self.PAD_ID] * (max_len - len(ids))

    def __len__(self):
        return len(self.token_to_id)


def make_clinical_triage_notes(n_per_class=90, seed=42):
    """Synthesize deterministic Routine (0) vs Urgent (1) notes."""
    rng = np.random.default_rng(seed)
    shared = ["patient", "ward", "reports", "clinic", "history"]
    routine = [
        "mild", "stable", "refill", "followup", "normal",
        "routine", "afebrile", "checkup", "rash", "cleared",
    ]
    urgent = [
        "severe", "chest", "pain", "dyspnea", "hypoxia",
        "sepsis", "tachycardia", "syncope", "bleeding", "emergency",
    ]
    texts, labels = [], []
    for _ in range(n_per_class):
        ctx = list(rng.choice(shared, size=2, replace=False))
        r_toks = list(rng.choice(routine, size=4, replace=False))
        u_toks = list(rng.choice(urgent, size=4, replace=False))
        if rng.random() < 0.2:
            r_toks[0] = str(rng.choice(urgent))
        if rng.random() < 0.2:
            u_toks[0] = str(rng.choice(routine))
        r_note = ctx + r_toks
        u_note = ctx + u_toks
        rng.shuffle(r_note)
        rng.shuffle(u_note)
        texts.append(" ".join(r_note))
        labels.append(0)
        texts.append(" ".join(u_note))
        labels.append(1)
    return texts, np.array(labels, dtype=np.int64)


class ClinicalNoteTransformer(nn.Module):
    """Mini Encoder Transformer for clinical triage classification."""

    def __init__(self, vocab_size, max_len=10, d_model=16, heads=2):
        super().__init__()
        self.emb = nn.Embedding(
            vocab_size, d_model, padding_idx=0
        )
        self.register_buffer(
            "pe", sinusoidal_positional_encoding(max_len, d_model)
        )
        self.enc = TransformerEncoderBlock(
            d_model, heads, d_ff=32, dropout=0.1
        )
        self.head = nn.Linear(d_model, 1)

    def forward(self, ids):
        pad_mask = (ids != 0).float()
        attn_mask = pad_mask.unsqueeze(1).unsqueeze(2)
        x = self.emb(ids) + self.pe[: ids.size(1)]
        h, attn = self.enc(x, mask=attn_mask)
        w = pad_mask.unsqueeze(-1)
        pooled = (h * w).sum(dim=1) / w.sum(dim=1).clamp(min=1.0)
        return self.head(pooled), attn

torch.manual_seed(42)
notes, y_all = make_clinical_triage_notes(n_per_class=90, seed=42)
tr_txt, va_txt, y_tr, y_va = train_test_split(
    notes, y_all, test_size=0.25, stratify=y_all, random_state=42
)
tok = ClinicalVocabTokenizer().fit(tr_txt)
X_tr = torch.tensor([tok.encode(t, 10) for t in tr_txt])
X_va = torch.tensor([tok.encode(t, 10) for t in va_txt])
y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

clf = ClinicalNoteTransformer(len(tok), max_len=10, d_model=16)
opt = torch.optim.Adam(clf.parameters(), lr=0.02)
loss_fn = nn.BCEWithLogitsLoss()
for _ in range(25):
    clf.train()
    opt.zero_grad()
    logits, _ = clf(X_tr)
    loss_fn(logits, y_tr_t).backward()
    opt.step()

clf.eval()
with torch.no_grad():
    va_logits, va_attn = clf(X_va)
    va_probs = torch.sigmoid(va_logits).squeeze(-1).numpy()
va_acc = float(((va_probs >= 0.5) == y_va).mean())
va_auc = float(roc_auc_score(y_va, va_probs))
print(f"Vocab size: {len(tok)} | Val Acc: {va_acc:.4f} | "
      f"Val AUC: {va_auc:.4f}")

# Inspect top attended tokens on an Urgent clinical note
sample = "patient reports severe chest pain dyspnea emergency"
s_ids = torch.tensor([tok.encode(sample, 10)])
with torch.no_grad():
    s_logit, s_attn = clf(s_ids)
    s_prob = float(torch.sigmoid(s_logit).item())
words = sample.split()
mean_recv = s_attn[0, :, : len(words), : len(words)].mean(dim=(0, 1))
ranked = sorted(
    zip(words, mean_recv.tolist()), key=lambda p: p[1], reverse=True
)
print(f"Urgent prob: {s_prob:.4f} | Top attended tokens:",
      [(w, round(sc, 3)) for w, sc in ranked[:4]])


## Section 32.3.6 — Parameter-Efficient Fine-Tuning With LoRA (`Delta W = B @ A`)

In [ ]:
class LoRALinear(nn.Module):
    """Freeze base nn.Linear and learn low-rank delta (B @ A)."""

    def __init__(self, base_linear, rank=4, alpha=8.0):
        super().__init__()
        self.base_linear = base_linear
        for p in self.base_linear.parameters():
            p.requires_grad = False
        self.rank = rank
        self.alpha = float(alpha)
        self.scaling = float(alpha) / rank
        in_f = base_linear.in_features
        out_f = base_linear.out_features
        self.lora_A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(out_f, rank))

    def forward(self, x):
        base_out = self.base_linear(x)
        delta = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + delta

torch.manual_seed(42)
base_proj = nn.Linear(64, 64)
orig_w = base_proj.weight.detach().clone()
lora_proj = LoRALinear(base_proj, rank=4, alpha=8.0)

x_adapt = torch.randn(16, 64)
y_target = torch.randn(16, 64)
with torch.no_grad():
    init_diff = (base_proj(x_adapt) - lora_proj(x_adapt)).abs().max()

base_n = sum(p.numel() for p in base_proj.parameters())
lora_n = sum(p.numel() for p in lora_proj.parameters() if p.requires_grad)
pct = 100.0 * lora_n / (base_n + lora_n)

opt_lora = torch.optim.Adam(
    [p for p in lora_proj.parameters() if p.requires_grad], lr=0.05
)
losses = []
for _ in range(15):
    opt_lora.zero_grad()
    loss = ((lora_proj(x_adapt) - y_target) ** 2).mean()
    loss.backward()
    opt_lora.step()
    losses.append(round(float(loss.item()), 4))

base_unchanged = bool(torch.equal(base_proj.weight, orig_w))
print(f"Step-0 max diff: {init_diff.item():.6f} | "
      f"Base frozen unchanged?: {base_unchanged}")
print(f"Base params: {base_n} | LoRA trainable: {lora_n} "
      f"({pct:.2f}% trainable)")
print(f"Adaptation MSE: start={losses[0]} -> end={losses[-1]}")


## Section 32.3.7 — Break It On Purpose! (Transformer Error Anatomy)

The next three cells intentionally trigger common Transformer bugs (`ValueError`, `IndexError`, and `RuntimeError`) so you recognize each traceback instantly in research code.

In [ ]:
# Bug #1: ValueError — d_model=30 is not divisible by num_heads=4
bad_mha = MultiHeadSelfAttention(d_model=30, num_heads=4)


In [ ]:
# Bug #2: IndexError — OOV token ID (>= vocab_size) in nn.Embedding
tiny_emb = nn.Embedding(num_embeddings=10, embedding_dim=8)
tiny_emb(torch.tensor([2, 12]))


In [ ]:
# Bug #3: RuntimeError — misaligned mask shape in masked_fill
bad_mask = torch.ones(1, 5)
scaled_dot_product_attention(Q_ex, K_ex, V_ex, mask=bad_mask)


## Section 32.4 — Stage 4 Blueprint: Clinical Triage Transformer + LoRA Domain Adaptation Audit

In [ ]:
def run_clinical_transformer_lora_audit(seed=42):
    """Train triage Transformer, wrap Q/V in LoRA, and audit params."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    texts, y = make_clinical_triage_notes(n_per_class=90, seed=seed)
    tr_t, va_t, y_tr, y_va = train_test_split(
        texts, y, test_size=0.25, stratify=y, random_state=seed
    )
    tok = ClinicalVocabTokenizer().fit(tr_t)
    X_tr = torch.tensor([tok.encode(t, 10) for t in tr_t])
    X_va = torch.tensor([tok.encode(t, 10) for t in va_t])
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

    model = ClinicalNoteTransformer(len(tok), max_len=10, d_model=16)
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(25):
        model.train()
        opt.zero_grad()
        logits, _ = model(X_tr)
        loss_fn(logits, y_tr_t).backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        va_logits, va_attn = model(X_va)
        va_probs = torch.sigmoid(va_logits).squeeze(-1).numpy()

    # Freeze entire model and attach rank-2 LoRA adapters to W_q & W_v
    for p in model.parameters():
        p.requires_grad = False
    model.enc.attn.W_q = LoRALinear(
        model.enc.attn.W_q, rank=2, alpha=4.0
    )
    model.enc.attn.W_v = LoRALinear(
        model.enc.attn.W_v, rank=2, alpha=4.0
    )
    with torch.no_grad():
        lora_logits, _ = model(X_va)
        parity = bool(
            (va_logits - lora_logits).abs().max().item() < 1e-6
        )
    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    return {
        "val_acc": round(float(((va_probs >= 0.5) == y_va).mean()), 4),
        "val_auc": round(float(roc_auc_score(y_va, va_probs)), 4),
        "attn_shape": tuple(va_attn.shape),
        "lora_trainable_params": int(train_p),
        "lora_trainable_pct": round(100.0 * train_p / total_p, 2),
        "step0_parity_ok": parity,
    }

print(run_clinical_transformer_lora_audit(seed=42))
